<a href="https://colab.research.google.com/github/eric20041027/Data_Mining/blob/main/notebooks/train_pubmedbert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PubMedBERT 5-Fold Training (Colab A100)

Trains `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` (renamed PubMedBERT) on the medical abstracts dataset, 5-fold StratifiedKFold, class-weighted CE, bf16.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU**.
2. Run cells in order.
3. Optional: mount Drive at the end to back up `outputs/bert_runs/` before the session expires.

## 1. Clone repo

In [1]:
import os, sys, shutil
REPO_DIR = '/content/Data_Mining'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/eric20041027/Data_Mining.git $REPO_DIR
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
# Persist HF cache on Drive if mounted later; for now use local Colab disk.
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('CWD:', os.getcwd())
print('Top-level:', sorted(os.listdir()))

Cloning into '/content/Data_Mining'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 5), reused 26 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 6.26 MiB | 20.53 MiB/s, done.
Resolving deltas: 100% (5/5), done.
CWD: /content/Data_Mining
Top-level: ['.git', '.gitignore', 'README.md', 'Rule.md', 'kaggle_testset.csv', 'kaggle_testset_submission.csv', 'kaggle_trainset.csv', 'notebooks', 'outputs', 'plan.md', 'src']


## 2. Install dependencies

In [2]:
!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "datasets>=2.20" "scikit-learn>=1.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.0 MB/s eta 0:00:00


In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
else:
    print('WARNING: no GPU detected — change runtime to A100 before training.')

CUDA available: True
Device: NVIDIA A100-SXM4-40GB
bf16 supported: True


## 3. Smoke test — 1 fold × 1 epoch on 64 samples
Confirms the pipeline runs end-to-end before launching the full sweep (~30 sec on A100).

In [4]:
!python src/train_bert.py --fold 0 --seed 42 --smoke --tag smoke_test

2026-05-21 01:41:06.814432: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-21 01:41:06.886554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 28.0/28.0 [00:00<00:00, 255kB/s]
config.json: 100% 385/385 [00:00<00:00, 4.04MB/s]
vocab.txt: 226kB [00:00, 12.7MB/s]
pytorch_model.bin: 100% 440M/440M [00:02<00:00, 156MB/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly

## 4. Full 5-fold sweep — PubMedBERT base, seed=42
Each fold ≈ 8–12 min on A100 → total ~50 min.

In [ ]:
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 42
for fold in range(5):
    cmd = (
        f'python src/train_bert.py '
        f'--model {MODEL} --fold {fold} --seed {SEED} '
        f'--epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 '
        f'--tag pubmedbert_base_seed{SEED}_fold{fold}'
    )
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_base_seed42_fold0


## 5. Summarise OOF Macro F1 across folds

In [ ]:
import json, glob, numpy as np, pandas as pd
runs = sorted(glob.glob('outputs/bert_runs/pubmedbert_base_seed42_fold*/metrics.json'))
rows = [json.load(open(p)) for p in runs]
df = pd.DataFrame(rows)
print(df[['fold', 'val_macro_f1', 'train_secs']])
print(f'\nMean OOF Macro F1: {df.val_macro_f1.mean():.4f} (std {df.val_macro_f1.std():.4f})')

## 6. Build final submission (BERT + overlap constraint)

In [ ]:
!python src/ensemble_predict.py \
    --bert-runs 'outputs/bert_runs/pubmedbert_base_seed42_fold*' \
    --tag pubmedbert_v1
print()
!ls -la outputs/submission_*.csv

## 7. Back up artefacts before session ends

**Option A — download a tarball locally:**

In [ ]:
!tar -czf bert_runs.tar.gz outputs/bert_runs outputs/submission_pubmedbert_v1.csv
from google.colab import files
files.download('bert_runs.tar.gz')
files.download('outputs/submission_pubmedbert_v1.csv')

**Option B — copy to Google Drive:**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/Kaggle_new_backup
# !cp -r outputs/bert_runs /content/drive/MyDrive/Kaggle_new_backup/
# !cp outputs/submission_pubmedbert_v1.csv /content/drive/MyDrive/Kaggle_new_backup/

## 8. Optional: extra seeds for ensemble
Uncomment to run additional seeds. Each adds ~50 min.

In [ ]:
# MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
# for SEED in (2024, 7):
#     for fold in range(5):
#         cmd = (
#             f'python src/train_bert.py '
#             f'--model {MODEL} --fold {fold} --seed {SEED} '
#             f'--epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 '
#             f'--tag pubmedbert_base_seed{SEED}_fold{fold}'
#         )
#         print('>>>', cmd)
#         rc = os.system(cmd)
#         assert rc == 0, f'seed {SEED} fold {fold} failed'